# Region Analysis — CODEX cHL

Unsupervised tissue region discovery using the trained CIM backbone.

**Pipeline:**
1. Load the trained CIM/VICReg backbone
2. Precompute sliding-window grid coordinates (shape-only HDF5 read)
3. Stream patches lazily from HDF5, embed in mini-batches
4. PCA → k-means for each k in `N_CLUSTERS_LIST`
5. Elbow / silhouette sweep to identify best k
6. Per-k: UMAP · spatial tissue map · cell-type composition

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────

WORK_DIR      = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/src/MCA/z_RUNS/CODEX_cHL_CIM_VICReg'
H5_PATH       = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/CODEX_cHL/CODEX_cHL.h5'
MARKERS_PATH  = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/CODEX_cHL/used_markers.txt'

REGION_PATCH_SIZE = 128   # spatial size of each region patch in pixels
STRIDE            = 64    # step between patches (50 % overlap)

N_CLUSTERS_LIST   = [8, 10, 12, 14, 16]   # k values to explore

PCA_COMPONENTS    = 64    # PCA dimensionality before clustering

# Batch size auto-scales with patch area to stay within CUDA tensor size limits.
# A 64px patch at batch=128 is the reference; larger patches need smaller batches.
_REF_PATCH = 64
_REF_BATCH = 128
BATCH_SIZE  = max(1, int(_REF_BATCH * (_REF_PATCH / REGION_PATCH_SIZE) ** 2))

UMAP_MAX_SAMPLES  = 30_000  # subsample for UMAP speed; None = all

N_JOBS   = 8     # CPU threads for numpy / sklearn / UMAP

SAVE_DIR = '../z_RUNS/region_analysis'

# ───────────────────────────────────────────────────────────────────────────
print(f'Patch size : {REGION_PATCH_SIZE}px   Batch size : {BATCH_SIZE}')


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# ── Limit CPU threads before numpy/sklearn are imported ────────────────────
for _var in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
             'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[_var] = str(N_JOBS)

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import numpy as np
import h5py
import torch
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import umap
import json

from threadpoolctl import threadpool_limits
threadpool_limits(N_JOBS)
print(f'CPU thread limit : {N_JOBS}')

from MCA.src.utils import load_checkpoint

SAVE_DIR = Path(SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'Output  : {SAVE_DIR.resolve()}')

## 1 · Load the trained backbone

In [ ]:
result   = load_checkpoint(WORK_DIR, device=DEVICE)
model    = result['model']
backbone = model.backbone
backbone.eval()
print('Backbone loaded.')

with torch.no_grad():
    dummy    = torch.zeros(1, 41, REGION_PATCH_SIZE, REGION_PATCH_SIZE, device=DEVICE)
    feat     = backbone(dummy)[0].squeeze(-1).squeeze(-1)
    FEAT_DIM = feat.shape[1]
    print(f'Feature dim for {REGION_PATCH_SIZE}×{REGION_PATCH_SIZE} patch: {FEAT_DIM}')

## 2 · Load HDF5 metadata (no pixel data yet)

In [ ]:
def decode(arr):
    return np.array([x.decode() if isinstance(x, bytes) else x for x in arr])

with h5py.File(H5_PATH, 'r') as f:
    all_marker_names = decode(f['marker_names'][:])
    cell_sample_ids  = decode(f['coords']['sample_id'][:])
    cell_dim1        = f['coords']['DIM1'][:].astype(int)
    cell_dim2        = f['coords']['DIM2'][:].astype(int)
    cell_annotations = decode(f['annotation'][:])
    unique_samples   = decode(f['sample_ids'][:])

with open(MARKERS_PATH) as fh:
    used_names = np.array([l.strip() for l in fh if l.strip()])

marker2idx     = {m: i for i, m in enumerate(all_marker_names)}
marker_indices = np.array([marker2idx[m] for m in used_names])

print(f'Markers in panel : {len(all_marker_names)}  |  Used: {len(marker_indices)}')
print(f'Cells in dataset : {len(cell_dim1):,}')
print(f'Samples ({len(unique_samples)}): {list(unique_samples)}')

## 3 · Precompute sliding-window grid coordinates

In [ ]:
ps       = REGION_PATCH_SIZE
all_meta = []   # (sample_id, y, x, img_H, img_W)

with h5py.File(H5_PATH, 'r') as f:
    for sid in unique_samples:
        H, W = f['data'][sid]['image'].shape[:2]
        for y in range(0, H - ps + 1, STRIDE):
            for x in range(0, W - ps + 1, STRIDE):
                all_meta.append((sid, y, x, H, W))

print(f'Total region patches : {len(all_meta):,}')
for sid in unique_samples:
    n = sum(1 for m in all_meta if m[0] == sid)
    print(f'  {sid}: {n:,} patches')

## 4 · Stream patches from HDF5 and embed

In [ ]:
# Build sample -> patch index mapping so we can process one sample at a time
sample_to_meta_indices = defaultdict(list)
for i, (sid, *_) in enumerate(all_meta):
    sample_to_meta_indices[sid].append(i)

@torch.no_grad()
def flush_batch(batch_list, backbone, device):
    arr   = np.array(batch_list, dtype=np.float32)
    t     = torch.from_numpy(arr).to(device)
    feats = backbone(t)
    if isinstance(feats, (tuple, list)):
        feats = feats[0]
    feats = feats.squeeze(-1).squeeze(-1)
    return F.normalize(feats, dim=1).cpu().numpy().astype(np.float16)


# Pre-allocate output array (avoids building a large list of arrays)
embeddings = np.zeros((len(all_meta), FEAT_DIM), dtype=np.float16)

with h5py.File(H5_PATH, 'r') as f:
    pbar = tqdm(total=len(all_meta), desc='Embedding patches',
                unit='patch', dynamic_ncols=True)

    for sid in unique_samples:
        # One sequential HDF5 read per sample — vastly faster than 1 read per patch
        image = f['data'][sid]['image'][:].astype(np.float32)[:, :, marker_indices]

        indices       = sample_to_meta_indices[sid]
        batch_patches = []
        batch_idx     = []

        for i in indices:
            _, y, x, H, W = all_meta[i]
            batch_patches.append(image[y : y + ps, x : x + ps, :].transpose(2, 0, 1))
            batch_idx.append(i)

            if len(batch_patches) == BATCH_SIZE:
                embeddings[batch_idx] = flush_batch(batch_patches, backbone, DEVICE)
                pbar.update(len(batch_patches))
                batch_patches, batch_idx = [], []

        if batch_patches:   # flush tail for this sample
            embeddings[batch_idx] = flush_batch(batch_patches, backbone, DEVICE)
            pbar.update(len(batch_patches))

    pbar.close()

print(f'Embeddings: {embeddings.shape}  dtype={embeddings.dtype}')
print(f'RAM: {embeddings.nbytes / 1024**2:.1f} MB')


## 5 · PCA

In [ ]:
pca     = PCA(n_components=PCA_COMPONENTS, random_state=42)
emb_pca = pca.fit_transform(embeddings.astype(np.float32))  # PCA needs float32
print(f'PCA explained variance ({PCA_COMPONENTS} components): {pca.explained_variance_ratio_.sum():.1%}')


## 6 · Elbow / silhouette sweep over k

In [ ]:
idx_sub = np.random.choice(len(emb_pca), min(10_000, len(emb_pca)), replace=False)
X_sub   = emb_pca[idx_sub]

inertia, sil = [], []
for k in tqdm(N_CLUSTERS_LIST, desc='k sweep'):
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=5, batch_size=2048)
    km.fit(X_sub)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_sub, km.labels_, metric='euclidean', sample_size=5000))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(N_CLUSTERS_LIST, inertia, 'o-')
axes[0].set(xlabel='k', ylabel='Inertia', title='Elbow')
axes[1].plot(N_CLUSTERS_LIST, sil, 'o-')
axes[1].set(xlabel='k', ylabel='Silhouette', title='Silhouette')
for ax in axes:
    ax.set_xticks(N_CLUSTERS_LIST)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'k_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = N_CLUSTERS_LIST[int(np.argmax(sil))]
print(f'Best silhouette: k={best_k}  ({max(sil):.3f})')

## 7 · Compute UMAP once (reused for all k)

In [ ]:
n_umap = min(UMAP_MAX_SAMPLES, len(embeddings)) if UMAP_MAX_SAMPLES else len(embeddings)
idx_u  = np.random.choice(len(embeddings), n_umap, replace=False)

reducer = umap.UMAP(
    n_components = 2,
    n_neighbors  = 30,
    min_dist     = 0.1,
    metric       = 'cosine',
    random_state = 42,
    n_jobs       = N_JOBS,
    verbose      = True,
)
umap_emb = reducer.fit_transform(embeddings[idx_u].astype(np.float32))  # float32 for UMAP
print(f'UMAP done  ({n_umap:,} points)')


## 8 · Per-k: cluster + visualise

For every k in `N_CLUSTERS_LIST`, results are saved to `SAVE_DIR/k_{k}/`.

In [ ]:
# ── Precompute patch_index (sid, y, x) -> patch position in all_meta ──────
patch_index = {(sid, y, x): i for i, (sid, y, x, H, W) in enumerate(all_meta)}
sample_dims = {}
for sid, y, x, H, W in all_meta:
    if sid not in sample_dims:
        sample_dims[sid] = (H, W)

# ── Reverse mapping: for each cell find which patches contain it ─────────
# A cell at (cy, cx) is inside patch (py, px) iff:
#   py <= cy < py+ps  and  px <= cx < px+ps
# Patches start at multiples of STRIDE, so py in {0, STRIDE, 2*STRIDE, ...}
# Number of containing patches per cell: at most (ps//STRIDE)^2 = 4 for ps=64, stride=32
# Total work: O(N_cells * 4)  vs old O(N_patches * N_cells) — >1000x faster

patch_to_cell_annotations = defaultdict(list)   # patch_idx -> [cell_type, ...]

for cidx in tqdm(range(len(cell_sample_ids)), desc='Mapping cells to patches (once)',
                 unit='cell', dynamic_ncols=True):
    sid = cell_sample_ids[cidx]
    if sid not in sample_dims:
        continue
    cy, cx     = int(cell_dim1[cidx]), int(cell_dim2[cidx])
    H, W       = sample_dims[sid]
    annotation = cell_annotations[cidx]

    # First grid-aligned y that could contain this cell
    py_start = (max(0, cy - ps + 1) // STRIDE) * STRIDE
    px_start = (max(0, cx - ps + 1) // STRIDE) * STRIDE

    for py in range(py_start, min(cy + 1, H - ps + 1), STRIDE):
        for px in range(px_start, min(cx + 1, W - ps + 1), STRIDE):
            pidx = patch_index.get((sid, py, px))
            if pidx is not None:
                patch_to_cell_annotations[pidx].append(annotation)

print(f'Mapped {len(cell_sample_ids):,} cells to {len(patch_to_cell_annotations):,} patches')


def run_for_k(k, emb_pca, umap_emb, idx_u, all_meta, save_root):
    out_dir = save_root / f'k_{k}'
    out_dir.mkdir(exist_ok=True)
    cmap = plt.cm.get_cmap('tab20', k)

    # ── k-means ──────────────────────────────────────────────────────────
    km = MiniBatchKMeans(
        n_clusters=k, random_state=42, n_init=10,
        batch_size=min(4096, len(emb_pca)), max_iter=300,
    )
    labels = km.fit_predict(emb_pca)
    sizes  = np.bincount(labels)
    print(f'k={k}  inertia={km.inertia_:.1f}')
    for c, n in enumerate(sizes):
        print(f'  C{c}: {n:,} ({n/len(labels):.1%})')

    # ── UMAP coloured by this k ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(umap_emb[:, 0], umap_emb[:, 1],
                    c=labels[idx_u], cmap=cmap,
                    vmin=-0.5, vmax=k - 0.5, s=3, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='Cluster', ticks=range(k))
    ax.set(title=f'UMAP  k={k}  (n={len(idx_u):,})', xlabel='UMAP 1', ylabel='UMAP 2')
    plt.tight_layout()
    plt.savefig(out_dir / 'umap.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Spatial tissue maps ───────────────────────────────────────────────
    n_samples = len(unique_samples)
    fig, axes = plt.subplots(1, n_samples, figsize=(7 * n_samples, 6), squeeze=False)
    for ax, sid in zip(axes[0], unique_samples):
        entries = [(i, y, x, H, W) for i, (s, y, x, H, W) in enumerate(all_meta) if s == sid]
        if not entries:
            ax.set_visible(False); continue
        _, _, _, img_H, img_W = entries[0]
        canvas = np.full((img_H, img_W, 4), [0.85, 0.85, 0.85, 1.0])
        for i, y, x, _, _ in entries:
            canvas[y : y + ps, x : x + ps] = np.array(cmap(int(labels[i])))
        ax.imshow(canvas, aspect='equal')
        ax.set_title(sid, fontsize=9)
        ax.axis('off')
    lp = [mpatches.Patch(color=cmap(c), label=f'C{c}') for c in range(k)]
    fig.legend(handles=lp, loc='lower center', ncol=min(k, 8), frameon=False, fontsize=8)
    fig.suptitle(f'Spatial region map  k={k}', fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(out_dir / 'spatial_map.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Cell-type composition (fast: O(N_patches) lookup) ─────────────────
    cluster_counts = defaultdict(Counter)  # cluster -> Counter of cell types
    for pidx, ann_list in patch_to_cell_annotations.items():
        cl = int(labels[pidx])
        cluster_counts[cl].update(ann_list)

    all_types   = sorted({ct for c in cluster_counts.values() for ct in c})
    composition = {}
    for cl in range(k):
        total = sum(cluster_counts[cl].values())
        composition[cl] = {ct: (cluster_counts[cl].get(ct, 0) / total if total else 0.0)
                           for ct in all_types}

    comp_df = pd.DataFrame(composition).T[all_types]

    fig, ax = plt.subplots(figsize=(14, 5))
    comp_df.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', legend=True, width=0.8)
    ax.set(xlabel='Cluster', ylabel='Cell-type fraction',
           title=f'Cell-type composition  k={k}')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
    ax.set_xticklabels([f'C{c}' for c in range(k)], rotation=0)
    plt.tight_layout()
    plt.savefig(out_dir / 'composition.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Per-sample spatial + pie ──────────────────────────────────────────
    n_rows = len(unique_samples)
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, 6 * n_rows),
                             gridspec_kw={'width_ratios': [3, 1]})
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for row, sid in enumerate(unique_samples):
        ax_map, ax_pie = axes[row, 0], axes[row, 1]
        entries = [(i, y, x, H, W) for i, (s, y, x, H, W) in enumerate(all_meta) if s == sid]
        if not entries:
            ax_map.set_visible(False); ax_pie.set_visible(False); continue
        _, _, _, img_H, img_W = entries[0]
        canvas = np.full((img_H, img_W, 4), [0.9, 0.9, 0.9, 1.0])
        counts_s = Counter()
        for i, y, x, _, _ in entries:
            cl = int(labels[i])
            canvas[y : y + ps, x : x + ps] = np.array(cmap(cl))
            counts_s[cl] += 1
        ax_map.imshow(canvas, aspect='equal')
        ax_map.set_title(sid, fontsize=10)
        ax_map.axis('off')
        pie_cls = sorted(counts_s)
        ax_pie.pie([counts_s[c] for c in pie_cls],
                   labels=[f'C{c}' for c in pie_cls],
                   colors=[cmap(c) for c in pie_cls],
                   autopct='%1.0f%%', startangle=90, textprops={'fontsize': 8})
        ax_pie.set_title('Cluster area\nfraction', fontsize=9)
    lp = [mpatches.Patch(color=cmap(c), label=f'C{c}') for c in range(k)]
    fig.legend(handles=lp, loc='lower center', ncol=min(k, 8), frameon=False, fontsize=8)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.savefig(out_dir / 'spatial_map_with_pie.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────
    print(f'  {"Cluster":<10} {"Dominant type":<22} {"Frac":>7}  {"N cells":>9}  {"N patches":>10}')
    print('  ' + '-' * 64)
    for cl in range(k):
        dom   = max(composition[cl], key=composition[cl].get)
        total = sum(cluster_counts[cl].values())
        print(f'  C{cl:<9} {dom:<22} {composition[cl][dom]:>6.1%}'
              f'  {total:>9,}  {int(np.sum(labels == cl)):>10,}')

    # ── Save JSON + CSV ───────────────────────────────────────────────────
    with open(out_dir / 'results.json', 'w') as f:
        json.dump({'k': k, 'inertia': float(km.inertia_),
                   'cluster_sizes': {int(c): int(n) for c, n in enumerate(sizes)},
                   'cluster_composition': {
                       int(cl): {ct: float(v) for ct, v in comp_df.loc[cl].items()}
                       for cl in range(k)}}, f, indent=2)
    pd.DataFrame(
        [(sid, y, x, H, W, int(labels[i]))
         for i, (sid, y, x, H, W) in enumerate(all_meta)],
        columns=['sample_id', 'y', 'x', 'img_H', 'img_W', 'cluster']
    ).to_csv(out_dir / 'assignments.csv', index=False)
    print(f'  -> saved to {out_dir}')
    return labels, comp_df


# ── Run for every k ──────────────────────────────────────────────────────
all_results = {}
for k in N_CLUSTERS_LIST:
    print(f'\n{"="*60}')
    print(f'  Running k = {k}')
    print(f'{"="*60}')
    labels, comp_df = run_for_k(
        k, emb_pca, umap_emb, idx_u, all_meta, SAVE_DIR
    )
    all_results[k] = {'labels': labels, 'comp_df': comp_df}


## 9 · Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(N_CLUSTERS_LIST, inertia, 'o-')
axes[0].set(xlabel='k', ylabel='Inertia', title='Elbow')
axes[1].plot(N_CLUSTERS_LIST, sil, 'o-')
axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.6, label=f'best k={best_k}')
axes[1].set(xlabel='k', ylabel='Silhouette', title='Silhouette')
axes[1].legend(fontsize=9)
for ax in axes:
    ax.set_xticks(N_CLUSTERS_LIST)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'k_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Output structure:')
print(f'  {SAVE_DIR}/')
print(f'  k_sweep.png')
for k in N_CLUSTERS_LIST:
    marker = '  <- best silhouette' if k == best_k else ''
    print(f'  k_{k}/   [umap.png  spatial_map.png  composition.png  results.json]{marker}')